# 11 - Larger synthetic scale benchmark

- **Purpose:** test local CPU runtime, memory, donor splitting, and train-only transforms at larger synthetic sizes.
- **Medium tier:** 64 donors x 100 cells = 6,400 cells; 256 features per view; six models; five seeds.
- **Stress tier:** 128 donors x 250 cells = 32,000 cells; 1,000 features per view; generation, splitting, and transforms only.
- **Data mode:** `synthetic`; no file is downloaded or read.
- **Evidence label:** experimental result for local scale, not biological result.

This notebook answers whether the current local pipeline can handle a larger controlled workload. It does not answer whether an approved human cohort has enough donors, paired cells, or statistical power.

In [1]:
import json
import platform
import resource
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd

import p22
from p22.data import make_donor_split
from p22.data.transforms import fit_train_only
from p22.runs.config import load_run_config
from p22.testing import make_synthetic_multimodal
from p22.training.seed_runner import run_all_seeds, summarize_across_seeds

CONFIG_PATH = p22.REPO_ROOT / 'configs' / 'scale_pilot.json'
OUTPUT_DIR = p22.REPO_ROOT / 'reports' / 'generated' / 'scale'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
config = load_run_config(CONFIG_PATH)
print('repository:', p22.REPO_ROOT)
print('python:', sys.version.split()[0])
print('platform:', platform.platform())
print('data mode:', config.data_mode)
print('approval blocked:', p22.approval_blocked())
print('config hash:', config.config_hash)

repository: /Users/anuragdani/Github/niw-eb1a/P22
python: 3.11.15
platform: macOS-26.5.2-arm64-arm-64bit
data mode: synthetic
approval blocked: True
config hash: 9f7b6ae5f7d0aa1572de35f50a0a01999cc4cb46900b801170b6bb2c6a574dc5


In [2]:
assert config.data_mode == p22.SYNTHETIC
assert p22.approval_blocked()
assert config.dataset['n_donors'] == 64
assert config.dataset['cells_per_donor'] == 100
assert config.dataset['n_features_a'] == 256
assert config.dataset['n_features_b'] == 256
print('medium cells:', config.dataset['n_donors'] * config.dataset['cells_per_donor'])
print('models:', [model.name for model in config.models])
print('seeds:', list(config.seeds))

medium cells: 6400
models: ['logistic_regression_view_a', 'logistic_regression_view_b', 'mlp_view_a', 'mlp_view_b', 'concat_fusion', 'gated_fusion']
seeds: [0, 1, 2, 3, 4]


In [3]:
def peak_rss_mb():
    value = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
    return value / (1024 * 1024) if sys.platform == 'darwin' else value / 1024

print('initial peak RSS MB:', round(peak_rss_mb(), 1))

initial peak RSS MB: 366.9


## Medium tier: full pipeline

Each seed regenerates data, splits donors, fits train-only transforms, trains all six baselines, and reads the test split once per model.

In [4]:
medium_started = time.perf_counter()
medium_runs = []
per_seed_seconds = {}
for seed in config.seeds:
    started = time.perf_counter()
    run = run_all_seeds(config, seeds=[seed], n_replicates=config.bootstrap['n_replicates'])[0]
    medium_runs.append(run)
    per_seed_seconds[str(seed)] = round(time.perf_counter() - started, 3)
    print(f'seed {seed}: {per_seed_seconds[str(seed)]} seconds')
medium_seconds = time.perf_counter() - medium_started
medium_peak_rss_mb = round(peak_rss_mb(), 1)
medium_summary = summarize_across_seeds(medium_runs)
print('medium total seconds:', round(medium_seconds, 3))
print('peak RSS MB:', medium_peak_rss_mb)

seed 0: 5.268 seconds


seed 1: 2.793 seconds


seed 2: 3.365 seconds


seed 3: 3.157 seconds


seed 4: 3.148 seconds
medium total seconds: 17.731
peak RSS MB: 553.0


In [5]:
medium_table = pd.DataFrame(
    [
        {
            'model': name,
            'accuracy_mean': values['accuracy']['mean'],
            'accuracy_sd': values['accuracy']['std'],
            'accuracy_interval': values['accuracy']['interval'],
            'n_seeds': values['accuracy']['n_seeds'],
        }
        for name, values in medium_summary.items()
    ]
).sort_values('model')
medium_table

,model,accuracy_mean,accuracy_sd,accuracy_interval,n_seeds
0,concat_fusion,0.887538,0.034788,"[0.8443439438775717, 0.9307329791993513]",5
1,gated_fusion,0.887231,0.031438,"[0.8481954282577091, 0.9262661102038292]",5
2,logistic_regression_view_a,0.888000,0.029436,"[0.8514509773638744, 0.9245490226361258]",5
3,logistic_regression_view_b,0.876308,0.033368,"[0.8348756893349595, 0.9177396952804252]",5
4,mlp_view_a,0.889538,0.029410,"[0.8530206523114797, 0.9260562707654431]",5
5,mlp_view_b,0.881385,0.027485,"[0.8472570611179522, 0.9155121696512785]",5


## Stress tier: data path only

This tier is intentionally larger than the full model pilot. It checks array generation, donor-held-out assignment, and train-only transforms without spending local resources on five-seed training at this size.

In [6]:
stress_spec = {
    'n_donors': 128,
    'cells_per_donor': 250,
    'n_features_a': 1000,
    'n_features_b': 1000,
    'n_classes': 3,
    'class_signal': 0.9,
    'donor_nuisance': 0.7,
    'noise': 1.0,
}
stress_started = time.perf_counter()
stress_data = make_synthetic_multimodal(**stress_spec, seed=91)
stress_split = make_donor_split(stress_data.donor_ids, labels=stress_data.labels, seed=91)
stress_holdout = np.concatenate([stress_split.val_index, stress_split.test_index])
stress_train_ids = stress_data.cell_ids[stress_split.train_index]
stress_holdout_ids = stress_data.cell_ids[stress_holdout]
stress_a = fit_train_only(stress_data.view_a, stress_data.cell_ids, stress_train_ids, stress_holdout_ids)
stress_b = fit_train_only(stress_data.view_b, stress_data.cell_ids, stress_train_ids, stress_holdout_ids)
stress_donor_sets = [set(stress_split.donors(name)) for name in ('train', 'val', 'test')]
stress_overlap = sum(len(left & right) for index, left in enumerate(stress_donor_sets) for right in stress_donor_sets[index + 1:])
stress_seconds = time.perf_counter() - stress_started
stress_result = {
    'cells': stress_data.n_cells,
    'donors': stress_data.n_donors,
    'features_a': stress_data.view_a.shape[1],
    'features_b': stress_data.view_b.shape[1],
    'split_donors': stress_split.summary['realized_donor_counts'],
    'split_cells': stress_split.summary['realized_cell_counts'],
    'donor_overlap': stress_overlap,
    'transformed_shapes': [list(stress_a.transform(stress_data.view_a[:2]).shape), list(stress_b.transform(stress_data.view_b[:2]).shape)],
    'seconds': round(stress_seconds, 3),
    'peak_rss_mb': round(peak_rss_mb(), 1),
}
print(json.dumps(stress_result, indent=2))

{
  "cells": 32000,
  "donors": 128,
  "features_a": 1000,
  "features_b": 1000,
  "split_donors": {
    "train": 76,
    "val": 26,
    "test": 26
  },
  "split_cells": {
    "train": 19000,
    "val": 6500,
    "test": 6500
  },
  "donor_overlap": 0,
  "transformed_shapes": [
    [
      2,
      1000
    ],
    [
      2,
      1000
    ]
  ],
  "seconds": 0.862,
  "peak_rss_mb": 1614.7
}


In [7]:
payload = {
    'benchmark': 'larger synthetic scale',
    'evidence_labels': ['experimental result', 'verified'],
    'medium': {
        'config': config.to_dict(),
        'config_hash': config.config_hash,
        'seeds': list(config.seeds),
        'seconds': round(medium_seconds, 3),
        'per_seed_seconds': per_seed_seconds,
        'peak_rss_mb': medium_peak_rss_mb,
        'summary': medium_summary,
    },
    'stress': stress_result,
    'limitations': list(config.limitations),
}
report_path = OUTPUT_DIR / 'scale_pilot_summary.json'
report_path.write_text(json.dumps(payload, indent=2, default=str) + '\n', encoding='utf-8')
print('wrote:', report_path.relative_to(p22.REPO_ROOT))

wrote: reports/generated/scale/scale_pilot_summary.json


In [8]:
checks = {
    'approval remains blocked': p22.approval_blocked(),
    'medium has five seeds': len(medium_runs) == 5,
    'medium has six model outcomes per seed': all(len(run.outcomes) == 6 for run in medium_runs),
    'medium donors never overlap': all(run.split_summary['donor_overlap'] == 0 for run in medium_runs),
    'medium test cells are held out once': all(run.cell_counts['test'] > 0 for run in medium_runs),
    'stress has no donor overlap': stress_result['donor_overlap'] == 0,
    'stress transforms preserve feature rows': stress_result['transformed_shapes'] == [[2, 1000], [2, 1000]],
    'stress outputs are finite': bool(np.isfinite(stress_data.view_a).all() and np.isfinite(stress_data.view_b).all()),
}
for name, passed in checks.items():
    print(f"{'PASS' if passed else 'FAIL'}: {name}")
assert all(checks.values()), 'scale gate not satisfied'
print('SCALE gate: PASS')

PASS: approval remains blocked
PASS: medium has five seeds
PASS: medium has six model outcomes per seed
PASS: medium donors never overlap
PASS: medium test cells are held out once
PASS: stress has no donor overlap
PASS: stress transforms preserve feature rows
PASS: stress outputs are finite
SCALE gate: PASS


## Interpretation

The medium tier measures local full-pipeline behavior at 6,400 cells. The stress tier measures whether the data path handles 32,000 cells and 1,000 features per view. These are engineering results only.

A real cohort still requires metadata preflight, donor/class counts, pairing rate, QC exclusions, a frozen estimand, and a professor-approved go/no-go decision.